In [13]:
import numpy as np
from scipy.linalg import lu, qr

def lr_eigs(A, n_iter=3000):
    """Approximation des valeurs propres via l'algorithme LR (LU)."""
    Ak = A.astype(float).copy()
    history = []

    for _ in range(n_iter):
        P, L, U = lu(Ak)
        Ak = np.linalg.solve(P, L @ U)
        Ak = U @ L
        history.append(np.diag(Ak).copy())

    off_diag_norm = np.linalg.norm(Ak - np.diag(np.diag(Ak)))
    return Ak, np.array(history), off_diag_norm


def qr_eigs(A, n_iter=30, tol=1e-10):
    """Approximation des valeurs propres via l'algorithme QR avec boucle while et test d'arret."""
    Ak = A.astype(float).copy()
    history = []
    it = 0
    
    while(1):
        Q, R = qr(Ak)
        Ak = R @ Q
        history.append(np.diag(Ak).copy())
        # Test d'arrêt: si la norme hors-diagonale est suffisamment petite 
        if np.linalg.norm(Ak - np.diag(np.diag(Ak)))  < tol or it >= n_iter:
            print(f"  [QR] Convergence atteinte à l'itération {it+1}, hors-diag = {np.linalg.norm(Ak - np.diag(np.diag(Ak))):.2e}")
            return
        
        it += 1
        
    off_diag_norm = np.linalg.norm(Ak - np.diag(np.diag(Ak)))
    return Ak, np.array(history), off_diag_norm


# Matrice 1
A = np.array([
    [2, -1/2, 0, -1/2],
    [0, 4, 0, 2],
    [-1/2, 0, 6, 1/2],
    [0, 0, 1, 9]
], dtype=float)

# Matrice 2
B = np.array([
    [-5, 0, 1/2, 1/2],
    [1/2, 2, 1/2, 0],
    [0, 1, 0, 1/2],
    [0, 1/4, 1/2, 3]
], dtype=float)

# # Matrice 1
# A = np.array([
#     [2, -1,  0],
#     [-1, 2, -1],
#     [0, -1, 2]
# ], dtype=float)

# # Matrice 2 (orthogonale)
# B = np.array([
#     [1 / np.sqrt(2),  1 / np.sqrt(2)],
#     [1 / np.sqrt(2), -1 / np.sqrt(2)]
# ], dtype=float)


for name, M in [("A", A), ("B", B)]:
    print(f"\n===== Matrice {name} =====")
    exact = np.sort(np.linalg.eigvals(M).real)
    print("Valeurs propres exactes:", exact)

    Ak_lr, hist_lr, off_lr = lr_eigs(M, n_iter=30)
    approx_lr = np.sort(np.diag(Ak_lr).real)
    print("\n[LU / LR]")
    print("diag(Ak) finale:", np.diag(Ak_lr))
    print("approx valeurs propres:", approx_lr)
    print("norme hors diagonale:", off_lr)

    Ak_qr, hist_qr, off_qr = qr_eigs(M, n_iter=30)
    approx_qr = np.sort(np.diag(Ak_qr).real)
    print("\n[QR]")
    print("diag(Ak) finale:", np.diag(Ak_qr))
    print("approx valeurs propres:", approx_qr)
    print("norme hors diagonale:", off_qr)



===== Matrice A =====
Valeurs propres exactes: [2.         4.02684475 5.80033442 9.17282083]

[LU / LR]
diag(Ak) finale: [20.95709339 -4.4143607   3.73099386 -1.4475424 ]
approx valeurs propres: [-4.4143607  -1.4475424   3.73099386 20.95709339]
norme hors diagonale: 45.78233902706382
  [QR] Convergence atteinte à l'itération 31, hors-diag = 2.24e+00


TypeError: cannot unpack non-iterable NoneType object